# Chapter 12a: Why Attention Is the Most Important Feature for LLMs

Chapter 12 showed *how* attention works. This chapter asks **why** it matters so much.

Three claims:
1. Attention solved the **information bottleneck** that killed RNNs
2. Attention makes every token **directly reachable** from every other token
3. Attention is **parallelizable**, which unlocked massive scaling

We'll prove each claim with numbers.

---
## Part 1: The Information Bottleneck

An RNN compresses the entire sentence into a **single hidden state vector**.

```
"The cat that I saw yesterday at the park near the river sat down"
  h1 → h2 → h3 → h4 → h5 → h6 → h7 → h8 → h9 → h10 → h11 → h12
                                                              ↑
                                            EVERYTHING must fit here
```

By the time the RNN reaches "sat", it needs to remember "cat" from 10 steps ago.
But the hidden state has fixed size (say, 256 dims). It's like trying to pour
a lake through a garden hose — information gets lost.

Let's measure this.

In [ ]:
import numpy as np

np.random.seed(42)

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Simulate an RNN: hidden state = tanh(W_h * h + W_x * x)
hidden_size = 8
embed_size = 4

W_h = np.random.randn(hidden_size, hidden_size) * 0.3
W_x = np.random.randn(hidden_size, embed_size) * 0.3

# 12 random "word embeddings"
sentence = [np.random.randn(embed_size) for _ in range(12)]
word_labels = ["The", "cat", "that", "I", "saw", "yesterday",
               "at", "the", "park", "near", "the", "river"]

# Run the RNN
h = np.zeros(hidden_size)
states = [h.copy()]
cat_embedding = None

print("RNN: How much of 'cat' (word 2) survives in later hidden states?\n")

for i, (x, word) in enumerate(zip(sentence, word_labels)):
    h = np.tanh(W_h @ h + W_x @ x)
    states.append(h.copy())
    if word == "cat":
        cat_state = h.copy()

# How similar is each later state to the state right after "cat"?
print(f"  {'Step':<6} {'Word':<12} {'Similarity to cat-state':>25} {'Bar'}")
print(f"  {'----':<6} {'----':<12} {'-------------------------':>25} {'---'}")
for i in range(1, len(states)):
    sim = cosine_sim(states[i], cat_state)
    bar = '#' * int(max(0, sim) * 40)
    marker = ' <-- cat' if i == 2 else ''
    word = word_labels[i-1] if i <= len(word_labels) else ''
    print(f"  {i:<6} {word:<12} {sim:>25.4f} {bar}{marker}")

print("\n  The signal from 'cat' fades as more words are processed.")
print("  This is the INFORMATION BOTTLENECK.")

---
## Part 2: Attention Gives Direct Access

In a transformer, word 12 doesn't need to reach word 2 through 10 intermediate steps.
It just **looks directly at it**.

```
RNN path:        word2 → h3 → h4 → h5 → ... → h12  (10 hops, signal decays)
Attention path:  word2 ←────────────────────── word12  (1 hop, direct)
```

Let's compute attention for a longer sentence and see how it maintains connections.

In [ ]:
# Self-attention: every word can directly attend to every other word
words = ["The", "cat", "that", "I", "saw", "yesterday", "sat", "down"]
n = len(words)
d_model = 6
d_k = 4

np.random.seed(7)

# Embeddings
X = np.random.randn(n, d_model) * 0.5
# Make 'cat' and 'sat' embeddings somewhat similar (both are about the cat acting)
X[1] = np.array([0.8, 0.1, 0.6, 0.3, 0.9, 0.2])  # cat
X[6] = np.array([0.7, 0.3, 0.5, 0.4, 0.8, 0.3])  # sat (similar to cat)

W_Q = np.random.randn(d_model, d_k) * 0.4
W_K = np.random.randn(d_model, d_k) * 0.4

Q = X @ W_Q
K = X @ W_K
scores = Q @ K.T / np.sqrt(d_k)
weights = softmax(scores)

print("Attention lets 'sat' (word 7) directly see 'cat' (word 2):\n")
print(f"  'sat' attention weights over all words:\n")
sat_idx = 6

# Sort by weight
order = np.argsort(-weights[sat_idx])
for rank, j in enumerate(order):
    w = weights[sat_idx, j]
    bar = '#' * int(w * 60)
    dist = abs(sat_idx - j)
    print(f"  {words[j]:>12}  {w:.3f}  {bar}  (distance={dist})")

print(f"\n  Distance between 'cat' and 'sat': {abs(sat_idx - 1)} words apart")
print(f"  In an RNN, that signal passes through {abs(sat_idx - 1)} hidden states.")
print(f"  In attention, it's a DIRECT connection — no decay.")

---
## Part 3: The Full Self-Attention Computation — Every Number

Let's trace the complete attention computation for a 3-word sentence,
showing every single intermediate value.

Sentence: **"I love cats"** with `d_model=4`, `d_k=3`.

In [ ]:
# Setup: 3 words, 4-dim embeddings, project to 3-dim Q/K/V
words = ["I", "love", "cats"]
d_model = 4
d_k = 3

X = np.array([
    [1.0, 0.0, 0.5, 0.2],   # I
    [0.3, 0.8, 0.1, 0.7],   # love
    [0.9, 0.2, 0.7, 0.1],   # cats
])

W_Q = np.array([
    [ 0.2,  0.1, -0.3],
    [ 0.4, -0.2,  0.1],
    [-0.1,  0.3,  0.5],
    [ 0.3,  0.2, -0.1],
])

W_K = np.array([
    [ 0.1,  0.3,  0.2],
    [-0.2,  0.1,  0.4],
    [ 0.3, -0.1,  0.1],
    [ 0.2,  0.4, -0.2],
])

W_V = np.array([
    [ 0.5, -0.1,  0.2],
    [ 0.1,  0.3, -0.2],
    [-0.3,  0.2,  0.4],
    [ 0.2, -0.1,  0.3],
])

print("INPUT EMBEDDINGS (X):")
print(f"  Each word is a {d_model}-dimensional vector\n")
for i, w in enumerate(words):
    print(f"  x_{w:>4} = [{', '.join(f'{v:>5.1f}' for v in X[i])}]")

print(f"\nWEIGHT MATRICES (learned during training):")
print(f"  W_Q: {W_Q.shape}  projects {d_model}→{d_k} dims")
print(f"  W_K: {W_K.shape}  projects {d_model}→{d_k} dims")
print(f"  W_V: {W_V.shape}  projects {d_model}→{d_k} dims")

In [ ]:
# STEP 1: Compute Q, K, V with full matrix multiply shown
print("=" * 65)
print("STEP 1: Compute Q = X @ W_Q  (Query for each word)")
print("=" * 65)

Q = X @ W_Q

for i, w in enumerate(words):
    print(f"\n  Q_{w} = x_{w} @ W_Q")
    print(f"       = [{', '.join(f'{v:>5.1f}' for v in X[i])}] @ W_Q")
    # Show each element of the result
    for j in range(d_k):
        terms = [f"{X[i,k]:>5.1f}*{W_Q[k,j]:>5.1f}" for k in range(d_model)]
        result = sum(X[i,k] * W_Q[k,j] for k in range(d_model))
        print(f"       [{j}] = {' + '.join(terms)} = {result:>6.3f}")
    print(f"       Q_{w} = [{', '.join(f'{v:>6.3f}' for v in Q[i])}]")

print(f"\n  Full Q matrix:")
for i, w in enumerate(words):
    print(f"    {w:>4}: [{', '.join(f'{v:>6.3f}' for v in Q[i])}]")

In [ ]:
K = X @ W_K
V = X @ W_V

print("K = X @ W_K  (Key for each word):")
for i, w in enumerate(words):
    print(f"    {w:>4}: [{', '.join(f'{v:>6.3f}' for v in K[i])}]")

print(f"\nV = X @ W_V  (Value for each word):")
for i, w in enumerate(words):
    print(f"    {w:>4}: [{', '.join(f'{v:>6.3f}' for v in V[i])}]")

In [ ]:
# STEP 2: Attention scores = Q @ K^T
print("=" * 65)
print("STEP 2: Attention Scores = Q @ K^T")
print("=" * 65)

raw_scores = Q @ K.T

print("\n  Each score = dot product of one Query with one Key")
print("  High dot product = 'these two words are relevant'\n")

for i, w_i in enumerate(words):
    for j, w_j in enumerate(words):
        terms = [f"{Q[i,k]:>6.3f}*{K[j,k]:>6.3f}" for k in range(d_k)]
        print(f"  score({w_i:>4},{w_j:>4}) = Q_{w_i} . K_{w_j} = {' + '.join(terms)} = {raw_scores[i,j]:>6.3f}")
    print()

print("Score matrix (raw):")
print(f"  {'':>6}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
for i, w in enumerate(words):
    print(f"  {w:>6}", end="")
    for j in range(len(words)):
        print(f" {raw_scores[i,j]:>8.3f}", end="")
    print()

In [ ]:
# STEP 3: Scale by sqrt(d_k)
print("=" * 65)
print(f"STEP 3: Scale by sqrt(d_k) = sqrt({d_k}) = {np.sqrt(d_k):.4f}")
print("=" * 65)

scale = np.sqrt(d_k)
scaled_scores = raw_scores / scale

print(f"\n  WHY scale? Without it, dot products grow with d_k.")
print(f"  Large values make softmax output nearly one-hot (all weight")
print(f"  on one word). Scaling keeps gradients healthy.\n")

print(f"  Before scaling vs After scaling (divide by {scale:.4f}):\n")
print(f"  {'':>6}", end="")
for w in words:
    print(f" {w+' raw':>10} {w+' scaled':>10}", end="")
print()
for i, w in enumerate(words):
    print(f"  {w:>6}", end="")
    for j in range(len(words)):
        print(f" {raw_scores[i,j]:>10.3f} {scaled_scores[i,j]:>10.3f}", end="")
    print()

In [ ]:
# STEP 4: Softmax
print("=" * 65)
print("STEP 4: Softmax (convert scores to probabilities)")
print("=" * 65)

weights = softmax(scaled_scores)

print("\n  softmax(x_i) = exp(x_i) / sum(exp(x_j))")
print("  Makes each row sum to 1.0 (probability distribution)\n")

for i, w_i in enumerate(words):
    row = scaled_scores[i]
    exps = np.exp(row - np.max(row))
    total = exps.sum()
    print(f"  Row '{w_i}':")
    print(f"    scaled scores: [{', '.join(f'{v:>6.3f}' for v in row)}]")
    print(f"    exp(scores):   [{', '.join(f'{v:>6.3f}' for v in exps)}]")
    print(f"    sum of exp:    {total:.3f}")
    print(f"    softmax:       [{', '.join(f'{v:>6.3f}' for v in weights[i])}]")
    print(f"    sum check:     {weights[i].sum():.3f}")
    print()

print("Attention weight matrix (who pays attention to whom):")
print(f"  {'':>6}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
for i, w in enumerate(words):
    print(f"  {w:>6}", end="")
    for j in range(len(words)):
        print(f" {weights[i,j]:>8.3f}", end="")
    print(f"  sum={weights[i].sum():.3f}")

In [ ]:
# STEP 5: Output = Weights @ V
print("=" * 65)
print("STEP 5: Output = Attention Weights @ V")
print("=" * 65)

output = weights @ V

print("\n  Each word's new representation = weighted sum of all Values")
print("  Words with higher attention weight contribute more.\n")

for i, w_i in enumerate(words):
    print(f"  output_{w_i} = ", end="")
    parts = []
    for j, w_j in enumerate(words):
        parts.append(f"{weights[i,j]:.3f} * V_{w_j}")
    print(" + ".join(parts))
    
    # Show the weighted vectors
    print(f"  {'':>12} = ", end="")
    weighted_parts = []
    for j in range(len(words)):
        wv = weights[i,j] * V[j]
        weighted_parts.append(f"[{', '.join(f'{v:>6.3f}' for v in wv)}]")
    print("\n                + ".join(weighted_parts))
    print(f"  {'':>12} = [{', '.join(f'{v:>6.3f}' for v in output[i])}]")
    print()

print("\nBefore attention (original X):")
for i, w in enumerate(words):
    print(f"  {w:>6}: [{', '.join(f'{v:>6.3f}' for v in X[i])}]   ({d_model} dims)")
print("\nAfter attention (context-enriched output):")
for i, w in enumerate(words):
    print(f"  {w:>6}: [{', '.join(f'{v:>6.3f}' for v in output[i])}]   ({d_k} dims)")
print("\nEach word now carries information from the words it attended to!")

---
## Part 4: Why Scaling Matters — A Demonstration

What happens if we DON'T scale by `sqrt(d_k)`?

With large `d_k`, dot products grow large. Large inputs to softmax push all the
probability mass onto one element, making attention nearly **one-hot**.
The model can only attend to one word at a time — it loses the ability to blend information.

In [ ]:
print("Effect of d_k on attention distribution (without scaling):\n")
print("  As d_k grows, dot products grow, softmax becomes more peaked.\n")

np.random.seed(42)
n_words = 5

for d_k_test in [2, 8, 32, 128, 512]:
    q = np.random.randn(d_k_test)
    K_test = np.random.randn(n_words, d_k_test)
    
    raw = q @ K_test.T
    unscaled = softmax(raw)
    scaled = softmax(raw / np.sqrt(d_k_test))
    
    print(f"  d_k = {d_k_test:>3}:")
    print(f"    Raw scores range: [{raw.min():>7.2f}, {raw.max():>7.2f}]")
    print(f"    Unscaled softmax: [{', '.join(f'{v:.3f}' for v in unscaled)}]  max={unscaled.max():.3f}")
    print(f"    Scaled softmax:   [{', '.join(f'{v:.3f}' for v in scaled)}]  max={scaled.max():.3f}")
    print()

---
## Part 5: Causal Masking — Why Autoregressive LLMs Need It

When generating text, the model predicts the **next** word.
It must NOT see future words — that would be cheating.

```
"I love cats ___"

To predict ___:
  'I'    can see: [I]
  'love' can see: [I, love]
  'cats' can see: [I, love, cats]
  '___'  can see: [I, love, cats]  → predict what comes next
```

This is done by setting future positions to `-inf` before softmax.
`softmax(-inf) = 0`, so future words get zero attention.

In [ ]:
words = ["I", "love", "cats"]
n = len(words)

# Reuse scaled_scores from Step 3
print("Causal masking step by step:\n")

print("1. Start with scaled attention scores:")
print(f"   {'':>6}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
for i, w in enumerate(words):
    print(f"   {w:>6}", end="")
    for j in range(n):
        print(f" {scaled_scores[i,j]:>8.3f}", end="")
    print()

# Create mask
mask = np.triu(np.ones((n, n)), k=1)  # upper triangle = future
print("\n2. Mask matrix (1 = future position, 0 = allowed):")
print(f"   {'':>6}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
for i, w in enumerate(words):
    print(f"   {w:>6}", end="")
    for j in range(n):
        print(f" {int(mask[i,j]):>8}", end="")
    print()

# Apply mask
masked = scaled_scores.copy()
masked[mask == 1] = -1e9

print("\n3. After masking (future → -inf):")
print(f"   {'':>6}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
for i, w in enumerate(words):
    print(f"   {w:>6}", end="")
    for j in range(n):
        if mask[i,j] == 1:
            print(f"     -inf", end="")
        else:
            print(f" {masked[i,j]:>8.3f}", end="")
    print()

# Softmax
causal_weights = softmax(masked)
print("\n4. After softmax (future positions → 0.000):")
print(f"   {'':>6}", end="")
for w in words:
    print(f" {w:>8}", end="")
print()
for i, w in enumerate(words):
    print(f"   {w:>6}", end="")
    for j in range(n):
        print(f" {causal_weights[i,j]:>8.3f}", end="")
    print(f"  sum={causal_weights[i].sum():.3f}")

print("\n  'I' puts 100% attention on itself (only word available).")
print("  'cats' splits attention across I, love, cats — but NEVER future words.")

---
## Part 6: Multi-Head Attention — Different Perspectives

One head might learn syntax (subject-verb agreement).
Another head might learn semantics (meaning similarity).
Another might learn position (nearby words).

By running multiple heads in parallel, the model captures **many types of relationships at once**.

In [ ]:
words = ["The", "big", "cat", "sat"]
n = len(words)
d_model = 6
d_k = 3
n_heads = 3

X = np.array([
    [0.1, 0.3, 0.0, 0.8, 0.2, 0.1],   # The
    [0.5, 0.7, 0.3, 0.2, 0.6, 0.4],   # big
    [0.8, 0.1, 0.6, 0.3, 0.9, 0.2],   # cat
    [0.2, 0.8, 0.3, 0.7, 0.1, 0.6],   # sat
])

print(f"Multi-Head Attention: {n_heads} heads, each sees different patterns\n")

np.random.seed(42)
all_head_weights = []

for h in range(n_heads):
    W_Q_h = np.random.randn(d_model, d_k) * 0.4
    W_K_h = np.random.randn(d_model, d_k) * 0.4
    
    Q_h = X @ W_Q_h
    K_h = X @ W_K_h
    scores_h = Q_h @ K_h.T / np.sqrt(d_k)
    weights_h = softmax(scores_h)
    all_head_weights.append(weights_h)
    
    labels = ["syntactic", "semantic", "positional"]
    print(f"  Head {h+1} ({labels[h]} patterns):")
    print(f"  {'':>6}", end="")
    for w in words:
        print(f" {w:>6}", end="")
    print()
    for i, w in enumerate(words):
        print(f"  {w:>6}", end="")
        for j in range(n):
            val = weights_h[i,j]
            if val > 0.35:
                print(f" {val:>5.3f}*", end="")
            else:
                print(f" {val:>6.3f}", end="")
        print()
    print()

print("  * = strongest attention in that row")
print("  Each head develops its own attention pattern.")
print("  The final output concatenates all heads, giving the model")
print("  access to ALL types of relationships simultaneously.")

---
## Part 7: Parallelism — Why GPUs Love Attention

RNNs must process tokens **sequentially** (each step depends on the previous).
Attention computes all pairs **at once** — it's just matrix multiplication.

```
RNN:        O(n) sequential steps     (can't parallelize)
Attention:  O(1) parallel steps       (one big matrix multiply)
            O(n^2) total operations   (but all in parallel!)
```

This is what made GPT-3 (175B parameters) possible. You can't train a model
that large with sequential processing — you'd wait forever.

In [ ]:
import time

print("Speed comparison: RNN vs Attention on different sequence lengths\n")
print("  RNN processes tokens one by one (sequential).")
print("  Attention does one matrix multiply (parallel on GPU).\n")

d = 64

print(f"  {'Seq Length':>10} {'RNN time':>12} {'Attn time':>12} {'RNN/Attn':>10}")
print(f"  {'─'*10} {'─'*12} {'─'*12} {'─'*10}")

for seq_len in [10, 50, 100, 500, 1000]:
    X_test = np.random.randn(seq_len, d)
    W_rnn = np.random.randn(d, d) * 0.1
    W_x_rnn = np.random.randn(d, d) * 0.1
    
    # RNN: sequential loop
    t0 = time.perf_counter()
    h = np.zeros(d)
    for t in range(seq_len):
        h = np.tanh(W_rnn @ h + W_x_rnn @ X_test[t])
    rnn_time = time.perf_counter() - t0
    
    # Attention: matrix multiply
    W_Q_test = np.random.randn(d, d) * 0.1
    W_K_test = np.random.randn(d, d) * 0.1
    t0 = time.perf_counter()
    Q_t = X_test @ W_Q_test
    K_t = X_test @ W_K_test
    scores_t = Q_t @ K_t.T / np.sqrt(d)
    attn_time = time.perf_counter() - t0
    
    ratio = rnn_time / (attn_time + 1e-10)
    print(f"  {seq_len:>10} {rnn_time*1000:>10.2f}ms {attn_time*1000:>10.2f}ms {ratio:>9.1f}x")

print("\n  On a GPU, the attention advantage is even more dramatic")
print("  because matrix multiply maps perfectly to GPU hardware.")
print("  RNNs can't be parallelized — each step needs the previous output.")

---
## Part 8: The Tradeoff — O(n^2) Memory and Compute

Attention's weakness: the score matrix is `n x n` (every token vs every token).

| Tokens | Score matrix size | Memory (float32) |
|--------|-------------------|------------------|
| 512    | 262K              | 1 MB             |
| 2048   | 4.2M              | 16 MB            |
| 8192   | 67M               | 256 MB           |
| 32768  | 1.07B             | 4 GB             |
| 131072 | 17.2B             | 64 GB            |

This is why context windows have limits, and why optimizations like
**Flash Attention**, **sparse attention**, and **KV caching** exist.

In [ ]:
print("Attention memory cost: O(n^2) per head\n")

print(f"  {'Context':>10} {'Score matrix':>15} {'Memory (fp32)':>15} {'Relative':>10}")
print(f"  {'─'*10} {'─'*15} {'─'*15} {'─'*10}")

base = None
for n_tok in [512, 2048, 4096, 8192, 32768, 131072]:
    elements = n_tok * n_tok
    memory_bytes = elements * 4  # float32
    if base is None:
        base = memory_bytes
    
    if memory_bytes < 1024**2:
        mem_str = f"{memory_bytes/1024:.0f} KB"
    elif memory_bytes < 1024**3:
        mem_str = f"{memory_bytes/1024**2:.0f} MB"
    else:
        mem_str = f"{memory_bytes/1024**3:.1f} GB"
    
    print(f"  {n_tok:>10,} {elements:>15,} {mem_str:>15} {memory_bytes/base:>9.0f}x")

print("\n  Double the context → 4x the memory (quadratic!).")
print("  This is why real models use optimizations:")
print("    - Flash Attention: recompute instead of storing")
print("    - KV Cache: store Key/Value for past tokens during generation")
print("    - Sparse Attention: only attend to nearby + selected tokens")

---
## Summary: Why Attention Is the Most Important Feature

| Problem Before Attention | How Attention Solves It |
|--------------------------|------------------------|
| Information bottleneck (RNN hidden state) | Every token has direct access to every other token |
| Long-range dependencies fade | Attention weight is computed directly, no decay |
| Sequential processing (slow training) | Matrix multiplication = massively parallel |
| Single relationship type per layer | Multi-head = multiple relationship types simultaneously |
| Can't scale beyond ~1B params | Parallelism enables 100B+ parameter models |

The complete formula that changed everything:

```
Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
```

One equation. Solved the bottleneck, enabled parallelism, unlocked scaling.
Without attention, there are no large language models.